In [4]:
import os
import time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Observatorio_Brecha_Digital_Colombia") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SESIÓN DE APACHE SPARK INICIADA")
print("Versión de Spark:", spark.version)
print("Modo de ejecución (Master):", spark.sparkContext.master)
print("Paralelismo por defecto (Particiones/Núcleos):", spark.sparkContext.defaultParallelism)

ruta_csv = "../data/processed/tic_limpio_2025.csv"
tamano_mb = os.path.getsize(ruta_csv) / (1024 * 1024)

df_spark = spark.read.csv(ruta_csv, header=True, inferSchema=True)

total_registros = df_spark.count()
num_particiones = df_spark.rdd.getNumPartitions()

print(f"\nArchivo procesado: {ruta_csv} ({tamano_mb:.2f} MB)")
print(f"Total de registros cargados en Spark: {total_registros:,}")
print(f"Total de columnas: {len(df_spark.columns)}")
print(f"Número de particiones distribuidas en Spark: {num_particiones}")

SESIÓN DE APACHE SPARK INICIADA
Versión de Spark: 4.2.1
Modo de ejecución (Master): local[*]
Paralelismo por defecto (Particiones/Núcleos): 12

Archivo procesado: ../data/processed/tic_limpio_2025.csv (43.55 MB)
Total de registros cargados en Spark: 221,043
Total de columnas: 37
Número de particiones distribuidas en Spark: 11


In [5]:
def ejecutar_analitica_spark(sdf):
    res_depto_zona = sdf.groupBy("NOMBRE_DEPARTAMENTO", "ZONA").agg(
        F.count("*").alias("Muestra_Personas"),
        F.round(F.sum("FEX_C"), 0).alias("Poblacion_Estimada"),
        F.round((F.sum(F.col("USA_INTERNET") * F.col("FEX_C")) / F.sum("FEX_C")) * 100, 2).alias("Pct_Usa_Internet"),
        F.round((F.sum(F.col("P1085S1") * F.col("FEX_C")) / F.sum("FEX_C")) * 100, 2).alias("Pct_Internet_Hogar"),
        F.round((F.sum(F.col("USA_COMPUTADOR") * F.col("FEX_C")) / F.sum("FEX_C")) * 100, 2).alias("Pct_Usa_Computador"),
        F.round((F.sum(F.col("USA_CELULAR") * F.col("FEX_C")) / F.sum("FEX_C")) * 100, 2).alias("Pct_Usa_Celular"),
        F.round(F.sum(F.col("PUNTAJE_CONECTIVIDAD") * F.col("FEX_C")) / F.sum("FEX_C"), 2).alias("Indice_Conectividad_Prom"),
        F.round(F.avg("PERCAPITA"), 0).alias("Ingreso_Percapita_Prom")
    ).orderBy(F.desc("Pct_Usa_Internet"))
    
    res_razones = sdf.filter(F.col("USA_INTERNET") == 0).groupBy("ZONA", "RAZON_NO_INTERNET").agg(
        F.count("*").alias("Muestra_Sin_Internet"),
        F.round(F.sum("FEX_C"), 0).alias("Poblacion_Sin_Internet")
    ).orderBy("ZONA", F.desc("Poblacion_Sin_Internet"))
    
    pdf_depto = res_depto_zona.toPandas()
    pdf_razones = res_razones.toPandas()
    return pdf_depto, pdf_razones

t0 = time.perf_counter()
df_1_part = df_spark.repartition(1)
_, _ = ejecutar_analitica_spark(df_1_part)
t_spark_1 = time.perf_counter() - t0

nucleos = spark.sparkContext.defaultParallelism
t0 = time.perf_counter()
df_n_part = df_spark.repartition(nucleos)
pdf_depto_zona, pdf_razones = ejecutar_analitica_spark(df_n_part)
t_spark_dist = time.perf_counter() - t0

speedup_spark = t_spark_1 / t_spark_dist
eficiencia_spark = (speedup_spark / nucleos) * 100

df_metricas_spark = pd.DataFrame([
    {"Configuración Spark": "Spark 1 Partición (Secuencial)", "Particiones/Núcleos": 1, "Tiempo (s)": round(t_spark_1, 4), "Speedup": 1.0, "Eficiencia (%)": 100.0},
    {"Configuración Spark": f"Spark Distribuido ({nucleos} Particiones)", "Particiones/Núcleos": nucleos, "Tiempo (s)": round(t_spark_dist, 4), "Speedup": round(speedup_spark, 4), "Eficiencia (%)": round(eficiencia_spark, 2)}
])

print("--- RENDIMIENTO APACHE SPARK (221,043 REGISTROS) ---")
display(df_metricas_spark)

print("\nTOP 10 DEPARTAMENTO / ZONA POR ACCESO A INTERNET")
display(pdf_depto_zona.head(10))

# Guardar tablas resumidas y métricas en evidencias y processed
df_metricas_spark.to_csv("../evidencias/capturas_cpd/metricas_spark.csv", index=False)
pdf_depto_zona.to_csv("../data/processed/resumen_brecha_depto_zona.csv", index=False, encoding="utf-8-sig")
pdf_razones.to_csv("../data/processed/resumen_razones_no_internet.csv", index=False, encoding="utf-8-sig")
print("\nTablas resumidas guardadas exitosamente.")

C:\Users\USUARIO\anaconda3\envs\BrechaDigital\lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
C:\Users\USUARIO\anaconda3\envs\BrechaDigital\lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
C:\Users\USUARIO\anaconda3\envs\BrechaDigital\lib\site-packages\pyspark\sql\pand

--- RENDIMIENTO APACHE SPARK (221,043 REGISTROS) ---


,Configuración Spark,Particiones/Núcleos,Tiempo (s),Speedup,Eficiencia (%)
0,Spark 1 Partición (Secuencial),1,3.2756,1.0000,100.00
1,Spark Distribuido (12 Particiones),12,11.9248,0.2747,2.29



TOP 10 DEPARTAMENTO / ZONA POR ACCESO A INTERNET


,NOMBRE_DEPARTAMENTO,ZONA,Muestra_Personas,Poblacion_Estimada,Pct_Usa_Internet,Pct_Internet_Hogar,Pct_Usa_Computador,Pct_Usa_Celular,Indice_Conectividad_Prom,Ingreso_Percapita_Prom
0,Valle del Cauca,Urbano,4014,3822823.0,90.26,87.08,43.09,85.49,3.06,1345782.0
1,Risaralda,Urbano,3104,775046.0,90.03,89.13,37.50,87.57,3.04,1423440.0
2,Atlántico,Urbano,4181,2577753.0,89.46,88.23,31.70,79.67,2.89,1173286.0
3,Santander,Urbano,4321,1760765.0,89.29,88.10,41.52,84.52,3.03,1460102.0
4,San Andrés y Providencia,Urbano,2016,41678.0,88.46,87.69,33.53,88.09,2.98,1798546.0
5,Norte de Santander,Urbano,3414,1258278.0,88.45,86.86,32.27,79.67,2.87,1017530.0
6,Boyacá,Urbano,3687,750936.0,87.20,84.88,43.08,85.34,3.01,1457967.0
7,Caldas,Urbano,3230,795113.0,87.14,86.38,39.92,87.08,3.01,1740815.0
8,Cundinamarca,Urbano,3603,2764994.0,86.86,84.30,43.81,85.55,3.01,1381018.0
9,Bolívar,Urbano,4490,1564859.0,86.77,79.45,34.50,77.30,2.78,1008635.0



Tablas resumidas guardadas exitosamente.
